<a href="https://colab.research.google.com/github/Nutchayapon/Super_AI_Engineer-SS.6/blob/main/Mini_Hackathon3_601661.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mini-Hackathon3 : FahMai RAG**

จัดทำโดย

รหัส: 601661

ชื่อ-นามสกุล: ณัชยภรณ์ คันธรส

# Setup

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters unstructured

In [ ]:
!pip install -q langchain-huggingface sentence-transformers

In [ ]:
import os, csv, re, time, requests
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Thai-llm

In [ ]:
THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="Openthaigpt", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/openthaigpt/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None

def parse_answer(text):
    # 1. หา ANSWER: ก่อนเลย — เพราะ LLM ถูก prompt ให้ตอบแบบนี้
    match = re.search(r'ANSWER:\s*\[?(\d+)\]?', text, re.IGNORECASE)
    if match:
        val = int(match.group(1))
        if 1 <= val <= 10:
            return val

    # 2. หาตัวเลขในวงเล็บเหลี่ยม [N] — มักเป็น summary ท้ายสุด
    matches = re.findall(r'\[(\d+)\]', text)
    if matches:
        val = int(matches[-1])  # เอาตัวสุดท้ายเผื่อมีหลายตัว
        if 1 <= val <= 10:
            return val

    # 3. fallback — หาตัวเลขท้ายสุด แต่ต้องอยู่ใกล้คำที่บ่งบอก
    match = re.search(
        r'(?:ตอบ|คำตอบ|เลือก|ข้อ|choice)[^\d]{0,10}([1-9]|10)',
        text, re.IGNORECASE
    )
    if match:
        return int(match.group(1))

    # 4. last resort — ตัวเลขตัวสุดท้ายของข้อความ
    numbers = re.findall(r'\b(10|[1-9])\b', text)
    if numbers:
        return int(numbers[-1])

    return None

# Data

In [ ]:
KAGGLE_API_KEY = userdata.get('KAGGLE_API')
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_KEY
!kaggle competitions download -c super-ai-engineer-s-6-fah-mai-rag-challenge-level-1
!unzip -q super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip -d data/

100% 321k/321k [00:00<00:00, 57.9MB/s]



# STEP 1 Data

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [ ]:
KB_DIR = "/content/data/data/knowledge_base" # Correcting the path for knowledge base

loader = DirectoryLoader(
    KB_DIR,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
    )
raw_documents = loader.load()

In [ ]:
print(f"Loaded {len(raw_documents)} documents from {KB_DIR}")

Loaded 118 documents from /content/data/data/knowledge_base


# STEP 2 Chunk

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
# 1. Define the specific Thai headers used in FahMai product pages.
# This ensures the splitter recognizes these 6 key sections.
headers_to_split_on = [
   ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3")
]

In [ ]:
# Initialize the Markdown Splitter with our specific headers
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False # Keep the header text inside the chunk for better LLM context
)

In [ ]:
# 2. Define the Recursive Splitter for secondary splitting
# We use a slightly larger chunk_size (800) to ensure Thai sentences stay together
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", "। ", " ", ""]  # เพิ่ม Thai full stop
)

In [ ]:
all_chunks = []

for doc in raw_documents:
    sections = markdown_splitter.split_text(doc.page_content)
    final_chunks = text_splitter.split_documents(sections)

    for chunk in final_chunks:      # ← อยู่ใน loop
        chunk.metadata.update(doc.metadata)
        source_name = Path(doc.metadata.get('source', '')).stem
        chunk.page_content = f"[สินค้า: {source_name}]\n" + chunk.page_content
        all_chunks.append(chunk)

print(f"Total processed chunks: {len(all_chunks)}")

# --- Verification ---
# Let's see if the metadata correctly captured our specific Thai headers
if len(all_chunks) > 0:
    print("\n[Sample Metadata]")
    print(all_chunks[0].metadata)
    print("\n[Sample Content Snippet]")
    print(all_chunks[0].page_content[:200])

Total processed chunks: 992

[Sample Metadata]
{'Header 1': 'นโยบายการคืนสินค้าและการขอเงินคืน — ร้านฟ้าใหม่', 'source': '/content/data/data/knowledge_base/policies/return_policy.md'}

[Sample Content Snippet]
[สินค้า: return_policy]
# นโยบายการคืนสินค้าและการขอเงินคืน — ร้านฟ้าใหม่  
**วันที่อัปเดต:** 1 มีนาคม 2569  
---


# STEP 3 Embedding and Vector DB

### Embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
model_name = "intfloat/multilingual-e5-base"


model_kwargs = {'device': 'cuda'} #GPU
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

### Vector DB

In [ ]:
# Install FAISS for GPU support
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.5 MB/s eta 0:00:00


In [ ]:
#Vector Store
vector_db = FAISS.from_documents(all_chunks, embeddings)
print("Vector Store created successfully with E5!")

Vector Store created successfully with E5!


# STEP 4  Retrieval
> Hybrid Search (BM25 + Vector) $\rightarrow$ RRF  $\rightarrow$ Top-k

In [ ]:
import langchain_community
print(langchain_community.__version__)
print(langchain_community.__file__)

0.4.1
/usr/local/lib/python3.12/dist-packages/langchain_community/__init__.py


### Setup Hybrid Retriever

In [ ]:
!pip install -q langchain-community rank-bm25 faiss-cpu

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [ ]:
bm25_retriever = BM25Retriever.from_documents(all_chunks)
bm25_retriever.k = 10

faiss_retriever = vector_db.as_retriever(search_kwargs={"k": 10})

# hybrid_retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6]   # weight
)

# STEP 5 Generation

In [ ]:
def expand_query(query):
    """เพิ่ม keyword สำคัญให้ query ก่อน retrieve"""
    expanded = query

    if any(w in query for w in ["ราคา", "เท่าไหร่", "บาท", "ค่า"]):
        expanded += " ราคา บาท"

    if any(w in query for w in ["ประกัน", "warranty", "เสีย", "ซ่อม"]):
        expanded += " การรับประกัน"

    if any(w in query for w in ["สเปค", "spec", "RAM", "กล้อง", "แบต"]):
        expanded += " สเปคสินค้า"

    return expanded

def get_hybrid_context(query, k=5):
    expanded = expand_query(query)  # expand
    docs = hybrid_retriever.invoke(expanded)

    context_xml = "<knowledge_base>\n"
    for i, doc in enumerate(docs[:k]):
        source = doc.metadata.get('source', 'Unknown')
        context_xml += f"  <document index='{i+1}' source='{source}'>\n"
        context_xml += f"    <content>{doc.page_content}</content>\n"
        context_xml += f"  </document>\n"
    context_xml += "</knowledge_base>"
    return context_xml

def build_xml_prompt(query, choices_text, context_xml):
    prompt = f"""คุณคือผู้ช่วยอัจฉริยะของร้าน "ฟ้าใหม่" (FahMai) ตอบคำถามโดยใช้ข้อมูลที่กำหนดให้เท่านั้น

ตัวอย่างการตอบ:
<example>
  <think>จากข้อมูล Watch S3 Ultra มีค่ากันน้ำ 10 ATM ตรงกับตัวเลือกที่ 3</think>
  ANSWER: [3]
</example>

{context_xml}

<question_details>
  <question>{query}</question>
  <choices>{choices_text}</choices>
</question_details>

<instructions>
  1. วิเคราะห์ข้อมูลใน knowledge_base อย่างละเอียด
  2. เลือกคำตอบที่ถูกต้องจากตัวเลือก (1-10)"
  3. หากไม่มีข้อมูลในฐานข้อมูล ให้ตอบ 9
  4. หากคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่เลย ให้ตอบ 10
  5. ขั้นตอน:
            1. ระบุว่าคำถามถามเรื่องอะไร
            2. หาข้อมูลที่ตรงกันใน Context
            3. ตรวจสอบทุกตัวเลือกก่อนสรุป 'ANSWER: X' เท่านั้น (โดย X คือหมายเลขตัวเลือก 1-10)สรุปด้วย ANSWER: [ตัวเลข] ในบรรทัดสุดท้าย
</instructions>

<response>
  <think>วิเคราะห์...</think>
  ANSWER: [ตัวเลข]
</response>"""
    return prompt

# STEP 6 LLM

In [ ]:
# Load questions from CSV
questions_df = pd.read_csv('/content/data/data/questions.csv')

# 1. Initialize the final list to store results
submission_list = []

print(f"🚀 Starting FULL RAG Pipeline with XML Prompting...")

# Iterate through ALL rows in the dataframe
for index, row in questions_df.iterrows():
    qid = row['id']
    query = row['question']

    # --- STEP 1: Format choices 1-10 into a readable string ---
    choices_list = []
    for i in range(1, 11):
        col_name = f'choice_{i}'
        choice_val = row[col_name]
        choices_list.append(f"{i}. {choice_val}")

    # Combine choices with newlines for the LLM prompt
    choices_text = "\n".join(choices_list)

    # --- STEP 2: Retrieval Phase (Hybrid Search) ---
    # Retrieve the top 4 most relevant document chunks in XML format
    context_xml = get_hybrid_context(query, k=10)

    # --- STEP 3: Build XML Prompt ---
    # Inject the query, choices, and context into the XML template
    full_prompt = build_xml_prompt(query, choices_text, context_xml)

    # --- STEP 4: LLM Generation Phase ---
    # Send the prompt to the ThaiLLM (e.g., Pathumma or Typhoon)
    messages = [{"role": "user", "content": full_prompt}]

    try:
        # Call the API
        raw_response = ask_llm(messages, model="pathumma")

        # --- STEP 5: Parsing Phase ---
        # Extract the specific choice number (1-10) from the AI's response
        # Using the improved regex parser discussed earlier is recommended here
        final_ans = parse_answer(raw_response)

    except Exception as e:
        # Handle API errors or unexpected crashes
        print(f"⚠️ Error processing Q{qid}: {e}")
        final_ans = None

    # --- STEP 6: Fallback Logic ---
    # If the AI response is unparseable or fails, default to Choice 9 (No Info)
    if final_ans is None:
        final_ans = 9

    # Append the result to the final list for export
    submission_list.append({'id': qid, 'answer': final_ans})

    # --- STEP 7: Real-time Logging ---
    # Display the full question and the final selected answer for monitoring
    print(f"📌 Q{qid}: {query}")
    print(f"✅ Final Answer: {final_ans}")
    print("-" * 50)

# --- STEP 8: Export to CSV ---
# Create a dataframe from the results and save it for submission
final_submission_df = pd.DataFrame(submission_list)
final_submission_df.to_csv('thaiGPT_final_submission.csv', index=False)

🚀 Starting FULL RAG Pipeline with XML Prompting...
📌 Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
✅ Final Answer: 5
--------------------------------------------------
📌 Q2: ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ
✅ Final Answer: 7
--------------------------------------------------
  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/openthaigpt/v1/chat/completions, retrying in 1s...
📌 Q3: หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ
✅ Final Answer: 2
--------------------------------------------------
📌 Q4: อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเครื่องใหม่ที่ฟ้าใหม่ได้ไหม?
✅ Final Answer: 6
--------------------------------------------------
📌 Q5: สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ
✅ Final Answer: 6
--------------------------------------------------
📌 Q6: ฟ้าใหม่จ่ายด้วย crypto ได้มั้ยคะ เช่น Bitcoin
✅ Final Answer: 8
--------------------------------------------------
📌 Q7: ซื้อ AirBook 14 มาแล้ว อยากรู้ว่าในกล่องมีอะไรมาบ้าง
✅ Final Answer: 1
--------------

# Ground Truth

In [ ]:
import pandas as pd

# 1. Ground Truth
gt_data = {
    'id': [1, 2, 3],
    'gt_choice': [5, 7, 2],
    'expected_text': ["10 ATM", "3,990 บาท", "Bluetooth 5.3"]
}
gt_df = pd.DataFrame(gt_data)

# 2. llm
# submission_list {'id': qid, 'answer': final_ans}
ai_results_df = pd.DataFrame(submission_list)

# 3. Merge
comparison = ai_results_df.merge(gt_df, on='id')
comparison['is_correct'] = comparison['answer'] == comparison['gt_choice']

accuracy = comparison['is_correct'].mean() * 100
print(f"📊 Accuracy: {accuracy:.2f}%")
print(comparison[['id', 'answer', 'gt_choice', 'is_correct']])

In [ ]:
print(raw_response)